# Pocket TTS Mooré: inference and voice cloning

Download one of the Mooré Pocket TTS models from the Hugging Face Hub, synthesize Mooré on **CPU and GPU**, and experiment with **voice cloning** from a reference recording downloaded from Google Drive.

| Model | Repo | Notes |
|---|---|---|
| `6l` | `<user>/pocket-tts-moore-6l` | 6-layer distilled student: fast on CPU, CFG built in (recommended) |
| `24l` | `<user>/pocket-tts-moore` | 24-layer teacher: ~3× slower, sampled without CFG |

**On Kaggle:**
1. *Settings → Accelerator*: a GPU (T4 / P100) for the GPU section (CPU works without one).
2. *Settings → Internet*: on.
3. If the model repo is private or gated, add your Hugging Face token under *Add-ons → Secrets* as `HF_TOKEN`.

Only clone voices you have permission to use.

In [ ]:
%pip install -q pocket-tts gdown

## 1. Settings

In [ ]:
HF_USERNAME = "Minervus00"
MODEL = "6l"                      # "6l" (distilled, fast) or "24l" (teacher)
REPOS = {
    "6l": f"{HF_USERNAME}/pocket-tts-moore-6l",
    "24l": f"{HF_USERNAME}/pocket-tts-moore",
}
LOCAL_MODEL_DIR = None            # or a local exported model folder (skips the Hub download)

# Reference voice: a Google Drive share link ("Anyone with the link") or a file ID ...
VOICE_DRIVE_URL = "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"
LOCAL_VOICE = None                # ... or a local audio file (used when set)
VOICE_MAX_SEC = 10.0              # the prompt is trimmed to this length (3-10 s of clean speech works best)

SENTENCES = [
    "La baa ne rẽ fãa, a Zehu ka sak Sẽn-Ka-Saab sẽn yaa Israyɛll Wẽnnaamã tõodã ne a sũur fãa ye.",
    "A Abimelɛk wãa tẽngã, n bao yamsem n fẽrg a tẽn-gãongã.",
    "La Amo-rãmbã sẽn yã tɩ Siiri nebã zoeeme wã, bãmb me zoe a Abisayi ne a tãb-biisã taoor n kẽ tẽngã pʋgẽ.",
    "Israyɛll neb wʋsg kii tãngã zugu.",
]
TEMPERATURE = 0.3
SEED = 0

from pathlib import Path
OUT_DIR = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("model:", MODEL, "| outputs:", OUT_DIR.resolve())

## 2. Hugging Face token (only needed for private or gated repos)

In [ ]:
import os

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF token:", "found" if token else "none (fine for public repos)")

## 3. Download the model

The repo's `moore.yaml` points at `hf://` paths; here we rewrite it to the downloaded local files so loading needs no second download.

In [ ]:
import re
from huggingface_hub import snapshot_download

if LOCAL_MODEL_DIR:
    model_dir = Path(LOCAL_MODEL_DIR)
else:
    model_dir = Path(snapshot_download(
        REPOS[MODEL], token=token,
        allow_patterns=["model.safetensors", "tokenizer.json", "*.yaml", "README.md"],
        ignore_patterns=["*_local.yaml", "training_args.yaml"],
    ))

config_src = next(p for p in sorted(model_dir.glob("*.yaml")) if not p.name.endswith("_local.yaml") and p.name != "training_args.yaml")
cfg = config_src.read_text()
cfg = re.sub(r"(?m)^weights_path:.*$", f"weights_path: {(model_dir / 'model.safetensors').resolve()}", cfg)
cfg = re.sub(r"(?m)^(\s*tokenizer_path:).*$", rf"\1 {(model_dir / 'tokenizer.json').resolve()}", cfg)
LOCAL_CONFIG = OUT_DIR / f"{MODEL}_local_config.yaml"
LOCAL_CONFIG.write_text(cfg)
n_layers = re.search(r"num_layers:\s*(\d+)", cfg).group(1)
print(f"model folder: {model_dir}\nconfig: {LOCAL_CONFIG} ({n_layers}-layer backbone)")

## 4. Reference voice from Google Drive

Share the file as *Anyone with the link*. Any common format works (wav, flac, mp3, ogg, m4a). The audio is converted to mono 16-bit WAV and trimmed to `VOICE_MAX_SEC`.

In [ ]:
import gdown
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def drive_file_id(url_or_id: str) -> str:
    for pattern in (r"/d/([\w-]{20,})", r"[?&]id=([\w-]{20,})"):
        m = re.search(pattern, url_or_id)
        if m:
            return m.group(1)
    return url_or_id.strip()  # already an ID

def load_audio(path):
    try:
        wav, sr = sf.read(path, dtype="float32", always_2d=True)
        return wav.mean(axis=1), sr
    except Exception:
        import librosa  # handles m4a/aac through ffmpeg
        wav, sr = librosa.load(path, sr=None, mono=True)
        return wav.astype("float32"), sr

if LOCAL_VOICE:
    voice_raw = Path(LOCAL_VOICE)
else:
    voice_raw = Path(gdown.download(id=drive_file_id(VOICE_DRIVE_URL), output=str(OUT_DIR / "voice_raw"), quiet=False))

wav, sr = load_audio(str(voice_raw))
print(f"downloaded {voice_raw.name}: {len(wav) / sr:.1f} s at {sr} Hz")

def make_prompt(seconds: float, name: str) -> Path:
    clip = wav[: int(seconds * sr)]
    clip = clip / max(1e-6, float(np.abs(clip).max())) * 0.9          # peak-normalize
    path = OUT_DIR / name
    sf.write(path, (clip * 32767).astype(np.int16), sr, subtype="PCM_16")
    return path

VOICE = make_prompt(VOICE_MAX_SEC, "voice_prompt.wav")
print(f"voice prompt: {VOICE} ({min(VOICE_MAX_SEC, len(wav) / sr):.1f} s)")
display(Audio(str(VOICE)))

## 5. Load the model and helpers

In [ ]:
import time
import torch
from pocket_tts import TTSModel

SAMPLE_RATE = 24000
_models = {}

def get_model(device: str, quantize: bool = False) -> TTSModel:
    key = (device, quantize)
    if key not in _models:
        model = TTSModel.load_model(config=str(LOCAL_CONFIG), temp=TEMPERATURE, quantize=quantize)
        model.to(device)
        _models[key] = model
    return _models[key]

def synthesize(model, voice_state, text, seed=SEED):
    torch.manual_seed(seed)
    if next(model.parameters()).is_cuda:
        torch.cuda.synchronize()
    start = time.perf_counter()
    audio = model.generate_audio(voice_state, text)
    if audio.is_cuda:
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    audio = audio.detach().float().cpu().numpy()
    return audio, len(audio) / SAMPLE_RATE, elapsed

def save(audio, name):
    path = OUT_DIR / name
    sf.write(path, (np.clip(audio, -1, 1) * 32767).astype(np.int16), SAMPLE_RATE, subtype="PCM_16")
    return path

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
print("devices:", DEVICES, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 6. CPU vs GPU

Same sentence, same seed, after one warm-up generation per device.
**RTF (real-time factor) = generation time / audio duration: RTF < 1 means faster than real time** (RTF 0.5 = 1 s of speech in 0.5 s).
Generation runs frame by frame, so a GPU helps less than you might expect, especially for the 6-layer model; the 24-layer model benefits more.

In [ ]:
rows = []
text = SENTENCES[1]
for device in DEVICES:
    model = get_model(device)
    state = model.get_state_for_audio_prompt(str(VOICE))
    synthesize(model, state, text)                         # warm-up
    audio, seconds, elapsed = synthesize(model, state, text)
    path = save(audio, f"{MODEL}_{device}.wav")
    rows.append((device, seconds, elapsed, path.name))
    print(f"{device}: {text}")
    display(Audio(audio, rate=SAMPLE_RATE))

print()
print(f"{'device':6} | {'audio (s)':>9} | {'time (s)':>8} | {'RTF':>5} | file")
for device, seconds, elapsed, name in rows:
    print(f"{device:6} | {seconds:9.2f} | {elapsed:8.2f} | {elapsed / seconds:5.2f} | {name}")

## 7. Voice-cloning experiments

### 7.1 All sentences in the cloned voice

In [ ]:
DEVICE = DEVICES[-1]   # GPU when available
model = get_model(DEVICE)
state = model.get_state_for_audio_prompt(str(VOICE))
for i, text in enumerate(SENTENCES):
    audio, seconds, elapsed = synthesize(model, state, text, seed=i)
    save(audio, f"{MODEL}_clone_{i:02d}.wav")
    print(f"[{i}] {seconds:.1f} s in {elapsed:.1f} s | {text}")
    display(Audio(audio, rate=SAMPLE_RATE))

### 7.2 Prompt length

Longer prompts usually match the voice better, up to ~10 s. Compare how similar each output sounds to the reference above.

In [ ]:
text = SENTENCES[0]
for seconds_prompt in (3, 5, 10):
    if seconds_prompt > len(wav) / sr + 0.5:
        print(f"{seconds_prompt} s: reference too short, skipped")
        continue
    prompt = make_prompt(seconds_prompt, f"voice_prompt_{seconds_prompt}s.wav")
    st = model.get_state_for_audio_prompt(str(prompt))
    audio, seconds, _ = synthesize(model, st, text)
    save(audio, f"{MODEL}_prompt_{seconds_prompt}s.wav")
    print(f"prompt {seconds_prompt} s -> {seconds:.1f} s of speech")
    display(Audio(audio, rate=SAMPLE_RATE))

### 7.3 Sampling temperature

Lower is steadier; higher is more varied and expressive, but more error-prone. The model's default is 0.3.

In [ ]:
text = SENTENCES[3]
for temp in (0.1, 0.3, 0.7):
    m = TTSModel.load_model(config=str(LOCAL_CONFIG), temp=temp)
    m.to(DEVICE)
    st = m.get_state_for_audio_prompt(str(VOICE))
    audio, seconds, _ = synthesize(m, st, text)
    save(audio, f"{MODEL}_temp_{temp}.wav")
    print(f"temperature {temp}: {seconds:.1f} s")
    display(Audio(audio, rate=SAMPLE_RATE))
    del m

### 7.4 Same text, different seeds

Sampling is random: the same prompt and text give slightly different renditions. Pick the best one, or use it to judge how stable the model is.

In [ ]:
text = SENTENCES[1]
for seed in (1, 2, 3):
    audio, seconds, _ = synthesize(model, state, text, seed=seed)
    save(audio, f"{MODEL}_seed_{seed}.wav")
    print(f"seed {seed}: {seconds:.1f} s")
    display(Audio(audio, rate=SAMPLE_RATE))

### 7.5 Your own text

In [ ]:
MY_TEXT = "Israyɛll neb wʋsg kii tãngã zugu."   # Mooré orthography; write numbers as words
import unicodedata
MY_TEXT = unicodedata.normalize("NFC", MY_TEXT)
audio, seconds, elapsed = synthesize(model, state, MY_TEXT)
save(audio, f"{MODEL}_my_text.wav")
display(Audio(audio, rate=SAMPLE_RATE))

## 8. Optional: int8 quantization on CPU

Smaller and usually faster on CPU. Compare the speed and listen for quality loss.

In [ ]:
qmodel = get_model("cpu", quantize=True)
qstate = qmodel.get_state_for_audio_prompt(str(VOICE))
synthesize(qmodel, qstate, SENTENCES[1])                    # warm-up
audio, seconds, elapsed = synthesize(qmodel, qstate, SENTENCES[1])
save(audio, f"{MODEL}_cpu_int8.wav")
print(f"cpu int8: {seconds:.1f} s in {elapsed:.2f} s -> RTF {elapsed / seconds:.2f} (< 1 = faster than real time)")
display(Audio(audio, rate=SAMPLE_RATE))

## 9. Download the outputs

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR.parent / f"pocket_tts_moore_{MODEL}_outputs"), "zip", OUT_DIR)
print("all generated audio:", archive)
sorted(p.name for p in OUT_DIR.glob("*.wav"))

## Tips

- **Voice prompt:** clean speech with no music or noise, one speaker, 3–10 s. The output copies the prompt's acoustics (noise, room, microphone).
- **Text:** Mooré orthography in Unicode NFC. Write numbers, dates and abbreviations as words: digits were removed from the training data.
- **Domain:** the models were trained on read Bible text from a few speakers, so expect a reading style.
- **Consent:** only clone voices you are allowed to use (Kyutai's Pocket TTS use policy applies).